# Experiment Diagnostics and Regression

This notebook demonstrates diagnostics for an experiment assignment and an adjusted binary-outcome model using the existing ShopFlow event table. It opens the SQLite database in read-only mode. Results are observational diagnostics on synthetic data; they do not establish real-world causal effects.

## Interpretation guardrails
- Count unique assigned users, not event rows, for sample-ratio checks.
- Check each user's group is consistent and each user belongs to one group.
- Pre-treatment covariates can be used for balance diagnostics and precision adjustment; do not adjust for post-treatment variables.
- Regression does not fix non-random assignment, missing outcomes, sample-ratio mismatch, peeking, or unmeasured confounding.
- If the dataset is simulated, any coefficient or p-value is a teaching example, not evidence about actual customers.

In [ ]:
import sys
from pathlib import Path

# Make repository imports work when Jupyter starts from the notebooks/ directory.
search_from = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (search_from, *search_from.parents)
     if (candidate / 'src' / 'experimentation.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository root containing src/experimentation.py')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import sqlite3
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.experimentation import holm_bonferroni, sample_ratio_mismatch

## 1. Read events without modifying the database

In [ ]:
db_path = (repo_root / 'data' / 'product_analytics.db').resolve()if not db_path.is_file():    raise FileNotFoundError(f'Database not found at expected repository path: {db_path}')with sqlite3.connect(f'{db_path.as_uri()}?mode=ro', uri=True) as connection:    events = pd.read_sql_query(        'SELECT user_id, timestamp, event_name, device, traffic_source, experiment_group FROM events',        connection,    )events.head()

## 2. Validate assignment consistency and sample ratio
This checks unique users and whether each user has one consistent group. If a user appears in multiple groups, stop and investigate the assignment data before interpreting treatment effects.

In [ ]:
user_group_counts = events.groupby('user_id')['experiment_group'].nunique(dropna=True)
multi_group_users = int((user_group_counts > 1).sum())
missing_group_users = int(events.loc[events['experiment_group'].isna(), 'user_id'].nunique())
assignment = (events.dropna(subset=['experiment_group'])
              .groupby('user_id')['experiment_group'].agg(lambda x: x.mode().iloc[0]))
group_counts = assignment.value_counts()
print(f'Users appearing in multiple groups: {multi_group_users:,}')
print(f'Users with at least one missing assignment event: {missing_group_users:,}')
print(group_counts)
if multi_group_users:
    raise ValueError('Assignment inconsistency detected; investigate before analyzing outcomes.')
if set(group_counts.index) != {'control', 'treatment'}:
    raise ValueError('Expected control and treatment groups; inspect observed labels.')
srm = sample_ratio_mismatch(int(group_counts['control']), int(group_counts['treatment']))
srm

A sample-ratio mismatch p-value below the chosen threshold is a warning, not a diagnosis. Investigate eligibility, randomization, logging, and exposure definitions. A non-significant test does not prove randomization was correct.

## 3. Build one row per user and inspect pre-treatment balance proxies
The event table may not identify which attributes were measured before assignment. Treat device and traffic source as covariates only if the product's assignment design confirms they are pre-treatment.

In [ ]:
user_features = (events.groupby('user_id', as_index=False)
    .agg(experiment_group=('experiment_group', lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else None),
         device=('device', lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else 'unknown'),
         traffic_source=('traffic_source', lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else 'unknown'),
         purchased=('event_name', lambda s: int((s == 'purchase').any()))))
user_features = user_features.dropna(subset=['experiment_group'])
balance = pd.crosstab(user_features['device'], user_features['experiment_group'], normalize='columns')
display(balance)

## 4. Logistic regression adjustment
The model estimates adjusted association between treatment assignment and whether a user had any purchase event. This is a simple teaching model; it does not account for cluster structure, pre-treatment eligibility, or uncertainty from the modeling process. If covariates are post-treatment, remove them.

In [ ]:
model_data = user_features[user_features['experiment_group'].isin(['control', 'treatment'])].copy()
model_data['treatment'] = (model_data['experiment_group'] == 'treatment').astype(int)
X = model_data[['treatment', 'device', 'traffic_source']]
y = model_data['purchased']
preprocess = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), ['device', 'traffic_source']),
    ('numeric', 'passthrough', ['treatment']),
])
model = Pipeline([('preprocess', preprocess), ('logistic', LogisticRegression(max_iter=1000))])
model.fit(X, y)
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = pd.DataFrame({'feature': feature_names, 'coefficient_log_odds': model.named_steps['logistic'].coef_[0]})
display(coefficients.sort_values('coefficient_log_odds', ascending=False))
print('Treatment coefficient is an adjusted log-odds association, not a percentage-point lift.')

## 5. Multiple testing and final reporting
Use Holm adjustment only for a pre-specified family of tests. The example below illustrates the calculation, not p-values from this dataset.


In [ ]:
example_p_values = [0.01, 0.04, 0.03]
pd.DataFrame({'raw_p_value': example_p_values, 'holm_adjusted_p_value': holm_bonferroni(example_p_values)})

## Reporting checklist
1. State the unit of randomization and primary outcome.
2. Report assigned users, exposures, missing outcomes, and group consistency.
3. Report sample-ratio diagnostics and investigate any imbalance before effect estimation.
4. Report conversion rates, absolute lift, confidence interval, p-value, planned sample size, and stopping rule.
5. Report guardrails and pre-specified multiplicity handling.
6. Explain why covariates are pre-treatment and disclose model limitations.
7. Label all simulated-data conclusions as illustrative, not real product evidence.